In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
# 1. Create local workspace
mkdir -p /content/KITTI_Project

# 2. Copy code and configs from Drive
cp -r /content/drive/MyDrive/KITTI_Project/src /content/KITTI_Project/
cp -r /content/drive/MyDrive/KITTI_Project/configs /content/KITTI_Project/
cp /content/drive/MyDrive/KITTI_Project/requirements.txt /content/KITTI_Project/

# 3. Copy and unzip data locally
mkdir -p /content/KITTI_Project/data
cp /content/drive/MyDrive/KITTI_Project/data/data.zip /content/KITTI_Project/data/
cp /content/drive/MyDrive/KITTI_Project/data/calib.zip /content/KITTI_Project/data/

unzip -q /content/KITTI_Project/data/data.zip -d /content/KITTI_Project/data/
unzip -q /content/KITTI_Project/data/calib.zip -d /content/KITTI_Project/data/

# 4. Flatten the nested KITTI folders (Pulling files out of subfolders)
mv /content/KITTI_Project/data/2011_09_26/* /content/KITTI_Project/data/ 2>/dev/null || true
mv /content/KITTI_Project/data/2011_09_26_drive_0064_sync/* /content/KITTI_Project/data/ 2>/dev/null || true

# 5. Final structure check
echo "--- Final Data Directory Structure ---"
ls -F /content/KITTI_Project/data/

# 6. Create output directory for experiment 0064
mkdir -p /content/KITTI_Project/output/exp_0064

--- Final Data Directory Structure ---
2011_09_26/
2011_09_26_drive_0064_sync/
calib_cam_to_cam.txt
calib_imu_to_velo.txt
calib_velo_to_cam.txt
calib.zip
data.zip
image_00/
image_01/
image_02/
image_03/
oxts/
velodyne_points/


In [ ]:
# Move LiDAR files out of the nested data/ subfolder
mv /content/KITTI_Project/data/velodyne_points/data/* /content/KITTI_Project/data/velodyne_points/ 2>/dev/null || true

# Move image files out of the nested data/ subfolder
mv /content/KITTI_Project/data/image_02/data/* /content/KITTI_Project/data/image_02/ 2>/dev/null || true

# Move oxts files out of the nested data/ subfolder
mv /content/KITTI_Project/data/oxts/data/* /content/KITTI_Project/data/oxts/ 2>/dev/null || true

# Verify that the actual files are now visible (you should see .bin or .png files)
echo "--- Top 3 LiDAR Files ---"
ls /content/KITTI_Project/data/velodyne_points/ | head -n 3

echo "--- Top 3 Image Files ---"
ls /content/KITTI_Project/data/image_02/ | head -n 3

In [3]:
%cd /content/KITTI_Project
!pip install -q -r requirements.txt

/content/KITTI_Project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 149.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 150.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 102.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
import yaml
import torch
import numpy as np
import os

from src.data.kitti_dataset import KittiDataset
from src.models.gaussian_model import GaussianModel
from src.models.loss_functions import combined_loss
from src.models.trainer import SplatTrainer
from src.models.densifier import Densifier

# Load Configuration for experiment 0064
with open("configs/residential_0064.yaml", "r") as f:
    config = yaml.safe_load(f)

print(f"Loaded configuration for experiment: {config['experiment']['name']}")

Loaded configuration for experiment: residential_0064


In [8]:
# 1. Initialize Dataset
dataset = KittiDataset(config)

# 2. Seed the Point Cloud using the first LiDAR frame
lidar_path = os.path.join(config['data']['lidar_dir'], "0000000000.bin")
scan = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 4)
pcd_xyz = scan[:, :3]

# 3. Initialize the Gaussian Model
model = GaussianModel(sh_degree=config['model']['sh_degree'])
model.create_from_pcd(pcd_xyz=pcd_xyz, device=config['experiment']['device'])

# 4. Initialize the Densifier
densifier = Densifier(
    model=model,
    grad_threshold=config['densification']['grad_threshold'],
    opacity_threshold=config['densification']['opacity_threshold'],
    extent=config['densification']['spatial_extent']
)

# 5. Initialize the Trainer
trainer = SplatTrainer(
    gaussian_model=model,
    dataset=dataset,
    densifier=densifier,
    iterations=config['training']['iterations'],
    device=config['experiment']['device']
)
print("Pipeline fully initialized and ready for training.")

Successfully loaded KITTI dataset indexing 570 frames.
Initialized 124244 Gaussians from point cloud.
Pipeline fully initialized and ready for training.


In [10]:
# Start the optimization loop
trainer.train()

Training Splats:   0%|          | 0/30000 [00:00<?, ?it/s]


AssertionError: block_width must be between 2 and 16

In [ ]:
# 1. Define output paths
local_out_path = os.path.join(config['experiment']['workspace'], "point_cloud.ply")
drive_out_path = f"/content/drive/MyDrive/KITTI_Project/output/exp_0064/point_cloud.ply"

# 2. Save the PLY file locally
model.save_ply(local_out_path)

# 3. Copy back to Google Drive
import shutil
shutil.copy2(local_out_path, drive_out_path)
print(f"Training complete! Point cloud successfully backed up to Google Drive at: {drive_out_path}")